# Week 1 Day 2 assignment: which free model is best with the agent?

The assignment in `labs/week1day2free.ipynb` asks which free model on OpenRouter gets the most claims right when it runs as the agent. This notebook answers that on all 25 claims, then puts the answer next to paid models so the free ones can be judged on cost and speed.

**Results at a glance** (all 25 claims, `TINY = False`; the table further down has the full data):

- **Best free model with the agent: `minimax/minimax-m3.1-flash:free`, 18 / 25**, 23 seconds a claim, $0. It's the only free model that gets every routine claim right and more than two hard ones. A second run by the contributor gave MiniMax 16 / 25 and `nemotron-3-ultra-550b-a55b:free` 17 / 25, so treat those two as level, with MiniMax faster.
- **Most accurate agent we tested: `deepseek-flash` with thinking on, 19 / 25**, at $0.039 for the 25 claims.
- **Cheapest model that scored 19 / 25: `deepseek-v4-flash-latest`**, $0.0078 for 25 claims. That is with a single call, not the agent.
- **No model got a manual claim right.** The manual claims need the claims manual or the customer's history, which the agent can't see yet, so the ceiling is 20 / 25.
- **`gpt-6-luna` is still the best value** in Ed's results: 17 / 25 as the agent for $0.0085, and 18 / 25 as a single call for $0.0042.

Every run is one pass over the queue. Free models are different from run to run, so a gap of one or two claims is noise.


## How the models were run

- **Agent**: exactly the lab's agent. LangChain `create_agent` with the policy wording in the system prompt and two tools, `get_policy` and `record_decision`. A claim with no recorded decision counts as wrong.
- **LLM**: the lab's single call with Structured Outputs. Not every model supports it: `minimax-m3.1-flash` returns a 404 because no provider accepts a JSON schema for it, so it has no LLM row.
- **Right** means the decision matches the handler's and the payout is within a dollar, the lab's `is_right`.
- **Cost** is for the 25 claims, from the token counts the API returned and the provider's list price. Free models cost $0. DeepSeek is billed at its peak-hour prices; off-peak is half.
- **Rows from Ed Donner** come from the results he showed for this lab. They are labelled, they have no per-difficulty breakdown, and I didn't re-run them.
- Models that couldn't be used (the free list changes, so some of these may work on another day): `nemotron-3-super-120b` (404 from NVIDIA); `gemma-4-31b-it`, `gemma-4-26b-a4b-it`, `xing4.0-29b-a4b` and `laguna-xs-2.1` (429, rate limited upstream when I tried); `inkling` (only works inside coding agents); and `lfm-2.5-2.6b`, which I stopped after 17 claims at 144 seconds a claim and 9 right. The 8 models in the table are the ones that finished all 25 claims.

## Setup

The imports, the connection and the data are the lab's own.


In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv(override=True)
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
print("OpenRouter API key found" if openrouter_api_key else "No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

In [ ]:
DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
claims, labels = get_sample_data(tiny=False)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

## The agent

The prompt, the tools and the scoring are copied from the lab. `run_agent` is the lab's agent loop with the model and the endpoint as arguments, so it runs a free model on OpenRouter or a model from another provider.


In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


def run_agent(model, claims, labels, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, extra_body=None):
    """Run the agent on each claim with one model, and score it against the handlers' decisions."""
    llm = ChatOpenAI(model=model, base_url=base_url, api_key=api_key, timeout=120, max_retries=5, extra_body=extra_body)
    agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
    results = []
    for claim in claims:
        start = time.time()
        recorded.pop(claim["claim_id"], None)
        try:
            agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
        except Exception as error:  # a model that can't call tools shows up as a wrong claim, not a crash
            print(f"{model}, {claim['claim_id']}: {str(error)[:100]}")
        answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0})
        label = labels[claim["claim_id"]]
        results.append({
            "model": model,
            "claim_id": claim["claim_id"],
            "difficulty": label["difficulty"],
            "right": is_right(answer, label),
            "seconds": round(time.time() - start, 1),
        })
    print(f"{model}: {sum(result['right'] for result in results)} of {len(results)} right")
    return results

## Running the free models

`RUN = False` by default, because all of these together make about 450 requests and free accounts are limited. With a little OpenRouter credit the limit is 1,000 free-model requests a day; without it, 50. Set `RUN = True` to run them yourself, or lower `LIMIT` to try a few claims.

Free models change and disappear, so check the list at `https://openrouter.ai/api/v1/models` (ids ending in `:free`, supporting `tools`) before you run it.

**To skip the slow models**, delete their lines from `FREE_MODELS` below: `nvidia/nemotron-3.5-lightning:free` (44 seconds a claim), `nvidia/nemotron-3.5-super-vl:free` (42) and `poolside/laguna-s-2.1:free` (68). The three that remain take about 23, 26 and 22 seconds a claim, so all 25 claims take roughly half an hour, against an hour and a half with all six.

In [ ]:
RUN = False
LIMIT = 25

FREE_MODELS = [
    "minimax/minimax-m3.1-flash:free",  # about 23 seconds a claim
    "nvidia/nemotron-3-ultra-550b-a55b:free",  # about 26
    "dots-studio/dots-3-note-preview:free",  # about 22
    # Slow: delete these three lines to test without them
    "nvidia/nemotron-3.5-lightning:free",  # about 44
    "nvidia/nemotron-3.5-super-vl:free",  # about 42
    "poolside/laguna-s-2.1:free",  # about 68
]

if RUN:
    free_results = []
    for model in FREE_MODELS:
        free_results += run_agent(model, claims[:LIMIT], labels)
    display(pd.DataFrame(free_results).pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum"))

### Other providers

`run_agent` takes a `base_url` and an `api_key`, so it also runs a model through its own provider. This is how the DeepSeek agent rows below were made: DeepSeek's API is OpenAI compatible, and its thinking is switched on with `extra_body`.

```python
run_agent(
    "deepseek-flash", claims, labels,
    base_url="https://api.deepseek.com",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    extra_body={"thinking": {"type": "enabled"}, "reasoning_effort": "low"},
)
```

The Haiku 5.5 row used the lab's single call through Anthropic's own SDK, with `effort` set to `low`.


## The results

`results.json` has every run from this comparison, with the Ed Donner rows labelled.


In [ ]:
results = pd.DataFrame(json.loads(Path("results.json").read_text(encoding="utf-8")))
results["accuracy"] = results["right"].astype(str) + " / " + results["claims"].astype(str)
results["cost"] = results["cost_usd"].map(lambda cost: "free" if cost == 0 else f"${cost:.4f}")
results["speed"] = results["seconds_per_claim"].map(lambda seconds: f"{seconds:.1f}s a claim")

table = results.sort_values(["right", "cost_usd"], ascending=[False, True])
display(table[["model", "technique", "reasoning", "accuracy", "cost", "speed", "provider"]].reset_index(drop=True))

### Accuracy by difficulty

Only for the runs made for this comparison.


In [ ]:
measured = results[results["source"] == "measured"].sort_values(["right", "cost_usd"], ascending=[False, True])
display(measured[["model", "technique", "routine", "hard", "manual"]].rename(columns={"routine": "routine (of 15)", "hard": "hard (of 5)", "manual": "manual (of 5)"}).reset_index(drop=True))

In [ ]:
chart = results.assign(label=results["model"] + " (" + results["technique"] + ", " + results["reasoning"] + ")")
fig = px.bar(
    chart.sort_values("right"),
    x="right", y="label", color="technique", orientation="h",
    hover_data=["cost", "speed", "provider"],
    title="Claims right out of 25",
)
fig.update_layout(height=600, yaxis_title=None, xaxis_title="Right out of 25")
fig.show()

### The best by each measure

Worked out from the table, with Ed's rows included.


In [ ]:
def show(title, row):
    display(Markdown(f"**{title}:** {row['model']} ({row['technique']}, {row['reasoning']}), {row['accuracy']}, {row['cost']}, {row['speed']}"))


agent = results[results["technique"] == "Agent"]
free = agent[agent["cost_usd"] == 0]
show("Best free model with the agent", free.sort_values(["right", "seconds_per_claim"], ascending=[False, True]).iloc[0])
show("Most accurate with the agent", agent.sort_values(["right", "cost_usd"], ascending=[False, True]).iloc[0])
show("Cheapest at 19 or more right", results[results["right"] >= 19].sort_values("cost_usd").iloc[0])
show("Cheapest paid agent at 17 or more right", agent[(agent["right"] >= 17) & (agent["cost_usd"] > 0)].sort_values("cost_usd").iloc[0])

## What this says

- **For free, use MiniMax M3.1 Flash.** It gets all 15 routine claims and 3 of 5 hard ones. The other free models lose a hard claim or two, or fail to record a decision. Nemotron nano-omni and Cohere north-mini-code mostly failed at calling tools.
- **Free costs time.** MiniMax took 23 seconds a claim, against 5 to 7 seconds for DeepSeek and Haiku. At queue sizes that matters more than the price, and the free limit would be hit quickly.
- **Thinking is what buys accuracy.** DeepSeek's agent went from 16 / 25 with thinking off to 19 / 25 with it on, entirely on the hard claims. It is also most of the output tokens, which is why it costs $0.039 against luna's $0.0085.
- **Haiku 5.5 did not pay off.** 16 / 25 for $0.0166 with a single call, which is about four times luna's cost for two fewer right. Its other runs (medium reasoning, and as the agent) scored no better, so they're not in the table.
- **The manual claims are still all wrong.** 0 / 5 for every model, as the lab says to expect. Closing that gap is a data problem, not a model one.
- **Caveats.** One run per row, and the free models change between runs, so a gap of one or two claims is not a result. A second run of the two leading free models scored 16 / 25 for MiniMax and 17 / 25 for Nemotron-ultra (18 and 16 here), which is why they're level. Prices are list prices on 21 October 2026. DeepSeek's cache hits are already in its cost.
